# Helpdesk loader — with role grouping (Song & Van der Aalst 2008)

Sibling of `Helpdesk_full_loader.ipynb` that reproduces the Camargo 2019 pre-processing step missing from the original loader: grouping raw resources into **roles** before feeding them into the LSTM.

### Algorithm (Song & Van der Aalst, *Analysis of Socio-Organizational Structures*, 2008)

1. Build a resource-by-activity frequency matrix `P` (how often each resource performs each activity).
2. Normalise each resource row to a probability distribution over activities.
3. Compute the Pearson correlation matrix between rows — a resource-to-resource *task similarity*.
4. Group resources whose pairwise correlation is ≥ `CORRELATION_THRESHOLD` (we use average-linkage agglomerative clustering on distance = `1 − correlation`).
5. Replace the raw `Resource` column with the discovered `Role` label.

The output pickle is written to `encoded_data/compare_camargo/helpdesk_all_5_roles_{train,test,val}.pkl` so the existing raw-resource pickles are left untouched.

Note: the resulting pickle has `categorical_columns = ['Activity', 'Role']`, so it is **not** compatible with the shipped `Helpdesk_camargo_act_1_suffix_length5.pkl` checkpoint (which was trained on raw Resource). You would retrain the Camargo model on this pickle to close the loader-side gap to the paper's reported 0.789 headline.

In [1]:
import importlib
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
import torch
import numpy as np
import pandas as pd


import event_log_loader.new_event_log_loader
importlib.reload(event_log_loader.new_event_log_loader)
from event_log_loader.new_event_log_loader import EventLogLoader

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

np.random.seed(17)

In [2]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
# --- Config ---
SRC_CSV = '../../../../../../../../data/helpdesk.csv'
ROLE_CSV = '../pkl/helpdesk_with_roles.csv'
OUT_DIR = '../pkl/'
RESULT_NAME = 'helpdesk_all'
SUFFIX_TAG = 'roles'  # appended so the role-based pickles sit next to the raw-resource ones

RESOURCE_COL = 'Resource'
ACTIVITY_COL = 'Activity'
ROLE_COL = 'Role'

CORRELATION_THRESHOLD = 0.85  # Song & Van der Aalst recommend ≥ 0.75; 0.85 is a common default

## 1. Discover roles

In [3]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
from sklearn.cluster import AgglomerativeClustering

raw = pd.read_csv(SRC_CSV)
print(f'Event log: {len(raw)} rows | cases: {raw["Case ID"].nunique()} | unique resources: {raw[RESOURCE_COL].nunique()}')

# 1 + 2: normalised resource × activity profile matrix
profile = raw.pivot_table(index=RESOURCE_COL, columns=ACTIVITY_COL, aggfunc='size', fill_value=0)
row_totals = profile.sum(axis=1).replace(0, 1)
profile_norm = profile.div(row_totals, axis=0)

# 3: Pearson correlation between resources
corr = profile_norm.T.corr()  # [resources, resources]
print(f'Resource profile matrix: {profile.shape}, correlation matrix: {corr.shape}')

# 4: agglomerative clustering with distance_threshold = 1 - correlation_threshold
#    -- average-linkage is equivalent to Song & Van der Aalst's clique-like grouping up to tie-breaking
dist = 1 - corr.clip(-1, 1).values
np.fill_diagonal(dist, 0.0)  # guard against floating noise
cluster = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=1 - CORRELATION_THRESHOLD,
    metric='precomputed',
    linkage='average',
)
labels = cluster.fit_predict(dist)

# 5: build resource -> role map, renaming clusters as Role_1..Role_K in descending population
resources = corr.index.tolist()
counts = pd.Series(labels, index=resources).value_counts()  # label -> # of resources in cluster
rename = {old: f'Role_{rank+1}' for rank, (old, _) in enumerate(counts.items())}
role_map = {r: rename[l] for r, l in zip(resources, labels)}

print(f'Discovered {len(set(role_map.values()))} roles (correlation threshold = {CORRELATION_THRESHOLD}):')
role_to_resources = {}
for r, role in role_map.items():
    role_to_resources.setdefault(role, []).append(r)
for role in sorted(role_to_resources, key=lambda x: -len(role_to_resources[x])):
    members = role_to_resources[role]
    print(f'  {role:<8s} ({len(members):2d} resources): {members[:8]}{" ..." if len(members) > 8 else ""}')

Event log: 21348 rows | cases: 4580 | unique resources: 22
Resource profile matrix: (22, 14), correlation matrix: (22, 22)
Discovered 7 roles (correlation threshold = 0.85):
  Role_1   (11 resources): ['Value 10', 'Value 11', 'Value 12', 'Value 13', 'Value 15', 'Value 16', 'Value 2', 'Value 4'] ...
  Role_2   ( 5 resources): ['Value 1', 'Value 14', 'Value 18', 'Value 19', 'Value 8']
  Role_3   ( 2 resources): ['Value 3', 'Value 5']
  Role_4   ( 1 resources): ['Value 17']
  Role_5   ( 1 resources): ['Value 20']
  Role_6   ( 1 resources): ['Value 21']
  Role_7   ( 1 resources): ['Value 22']


## 2. Write role-augmented CSV

EventLogLoader consumes a CSV path, so we persist the augmented table (raw columns + new `Role`) to disk and point the loader at it.

In [4]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
raw[ROLE_COL] = raw[RESOURCE_COL].map(role_map).fillna('Role_unknown')

from pathlib import Path
Path(ROLE_CSV).parent.mkdir(parents=True, exist_ok=True)
raw.to_csv(ROLE_CSV, index=False)
print(f'Wrote {ROLE_CSV} | {len(raw)} rows | role distribution:')
print(raw[ROLE_COL].value_counts().to_string())

Wrote ../../../../encoded_data/compare_camargo/helpdesk_with_roles.csv | 21348 rows | role distribution:
Role
Role_1    11418
Role_2     5166
Role_3     4609
Role_4       87
Role_6       34
Role_5       24
Role_7       10


## 3. Run EventLogLoader with `Role` instead of `Resource`

In [5]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
event_log_properties = {
    'case_name': 'Case ID',
    'concept_name': ACTIVITY_COL,
    'timestamp_name': 'Complete Timestamp',
    'date_format': '%Y/%m/%d %H:%M:%S.%f',
    'time_since_case_start_column': 'case_elapsed_time',
    'time_since_last_event_column': 'event_elapsed_time',
    'day_in_week_column': 'day_in_week',
    'seconds_in_day_column': 'seconds_in_day',
    'min_suffix_size': 5,
    'train_validation_size': 0.15,
    'test_validation_size': 0.2,
    'window_size': 'auto',
    'categorical_columns': [ACTIVITY_COL, ROLE_COL],
    'continuous_columns': ['case_elapsed_time'],
    'continuous_positive_columns': [],
}

event_log_loader = EventLogLoader(ROLE_CSV, event_log_properties)
print('window_size:', event_log_loader.encoder_decoder.window_size)

window_size: 18


In [6]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
train_dataset = event_log_loader.get_dataset('train')
path = f"{OUT_DIR}{RESULT_NAME}_{event_log_loader.encoder_decoder.min_suffix_size}_{SUFFIX_TAG}_train.pkl"
torch.save(train_dataset, path)
print(f'Saved {path}')
print(train_dataset.all_categories)

categorical tensors:   0%|          | 0/2 [00:00<?, ?it/s]

Activity:   0%|          | 0/2977 [00:00<?, ?it/s]

Role:   0%|          | 0/2977 [00:00<?, ?it/s]

continouous tensors:   0%|          | 0/1 [00:00<?, ?it/s]

case_elapsed_time:   0%|          | 0/2977 [00:00<?, ?it/s]

Saved ../../../../encoded_data/compare_camargo/helpdesk_all_5_roles_train.pkl
([('Activity', 16, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'EOS': 5, 'INVALID': 6, 'Insert ticket': 7, 'RESOLVED': 8, 'Require upgrade': 9, 'Resolve SW anomaly': 10, 'Resolve ticket': 11, 'Schedule intervention': 12, 'Take in charge ticket': 13, 'VERIFIED': 14, 'Wait': 15}), ('Role', 9, {'EOS': 1, 'Role_1': 2, 'Role_2': 3, 'Role_3': 4, 'Role_4': 5, 'Role_5': 6, 'Role_6': 7, 'Role_7': 8})], [('case_elapsed_time', 1, {})])


In [7]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
test_dataset = event_log_loader.get_dataset('test')
path = f"{OUT_DIR}{RESULT_NAME}_{event_log_loader.encoder_decoder.min_suffix_size}_{SUFFIX_TAG}_test.pkl"
torch.save(test_dataset, path)
print(f'Saved {path}')

categorical tensors:   0%|          | 0/2 [00:00<?, ?it/s]

Activity:   0%|          | 0/916 [00:00<?, ?it/s]

Role:   0%|          | 0/916 [00:00<?, ?it/s]

continouous tensors:   0%|          | 0/1 [00:00<?, ?it/s]

case_elapsed_time:   0%|          | 0/916 [00:00<?, ?it/s]

Saved ../../../../encoded_data/compare_camargo/helpdesk_all_5_roles_test.pkl


In [8]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
val_dataset = event_log_loader.get_dataset('val')
path = f"{OUT_DIR}{RESULT_NAME}_{event_log_loader.encoder_decoder.min_suffix_size}_{SUFFIX_TAG}_val.pkl"
torch.save(val_dataset, path)
print(f'Saved {path}')

categorical tensors:   0%|          | 0/2 [00:00<?, ?it/s]

Activity:   0%|          | 0/687 [00:00<?, ?it/s]

Role:   0%|          | 0/687 [00:00<?, ?it/s]

continouous tensors:   0%|          | 0/1 [00:00<?, ?it/s]

case_elapsed_time:   0%|          | 0/687 [00:00<?, ?it/s]

Saved ../../../../encoded_data/compare_camargo/helpdesk_all_5_roles_val.pkl


### Next step

Re-run `src/reimplemented_comparable_approaches/camargo_LSTM_suffix_pred/notebooks/training/Helpdesk/train_camargo_LSTM.ipynb` **pointing it at the new `_roles_` pickles** (swap the `helpdesk_all_5_train.pkl` / `_test` / `_val` paths for their `_roles_` variants, and update `model_feat` to `[['Activity', 'Role'], ['case_elapsed_time']]`). That produces a Camargo Helpdesk checkpoint on the paper's intended features; the mismatch to the 0.789 headline should shrink to just the architecture choice (`shared_cat` vs `FullShared_Join_LSTM`).

Threshold tuning: `CORRELATION_THRESHOLD = 0.85` yields the typical 4–6 roles reported in the literature. Lower thresholds merge more resources; the config cell above makes it easy to sweep.